|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>Observability<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: six numbers, and what each one is asking you to fix<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Compute the six numbers from a recorded run, then use them to answer a
question a single latency number cannot.

All arithmetic on arrays. The interesting part is Exercise 3.

In [ ]:
### run this cell: one server run, already recorded
NUM_REQUESTS = 4000
arrivals = np.cumsum(rng.exponential(1/25, size=NUM_REQUESTS))
queue_wait = rng.exponential(0.8, size=NUM_REQUESTS)          # the wait before the first step
output_lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=NUM_REQUESTS).astype(int) + 1
token_interval = rng.lognormal(mean=np.log(0.012), sigma=0.4, size=NUM_REQUESTS)  # s for each token
prefill_s = 0.04 * rng.lognormal(mean=np.log(400), sigma=0.6, size=NUM_REQUESTS)/1000
first_token = arrivals + queue_wait + prefill_s
finish = first_token + output_lengths*token_interval
print(f'{NUM_REQUESTS} requests over {finish.max():.0f} seconds')

# Exercise 1: the metrics

TTFT, TPOT, end-to-end latency, throughput, queue wait. Percentiles, not
means.

In [ ]:
# TTFT is from the arrival to the FIRST token. So it holds the queue wait AND
# the prefill. TPOT is the time for each token after that. Latency is all of it.
ttft = 
tpot = 
latency = 
total_tokens = output_lengths.sum()
wall = finish.max()

def percentile(values, rank):
  return float(np.percentile(values, rank))

print(f"{'metric':<12} {'p50':>9} {'p90':>9} {'p99':>9}")
for name, values in (('TTFT (s)', ttft), ('TPOT (ms)', tpot*1000),
                     ('latency (s)', latency)):
  print(f'{name:<12} {percentile(values,50):>9.3f} {percentile(values,90):>9.3f} '
        f'{percentile(values,99):>9.3f}')
print(f'\nthroughput  {total_tokens/wall:8.0f} tokens/s, {NUM_REQUESTS/wall:.1f} requests/s')
print(f'queue wait  {np.median(queue_wait):8.3f} s median')

# Exercise 2: goodput against three promises

A request is good only if it met **both** its TTFT and TPOT targets. Count
those, and compare with raw throughput.

In [ ]:
def goodput(ttft_sla, tpot_sla):
  """-> (the fraction of requests that met BOTH promises, the rate of them)."""
  met = 
  return met.mean(), met.sum()/wall

print(f"{'TTFT SLA':>9} {'TPOT SLA':>10} {'met':>7} {'goodput req/s':>14}")
for ttft_sla, tpot_sla in ((1.0, 0.020), (2.0, 0.030), (5.0, 0.050)):
  fraction_met, goodput_rate = goodput(ttft_sla, tpot_sla)
  print(f'{ttft_sla:>8.1f}s {tpot_sla*1000:>9.0f}ms {100*fraction_met:>6.1f}% '
        f'{goodput_rate:>14.1f}')
print(f'\nraw throughput {NUM_REQUESTS/wall:.1f} requests/s, and it does not change')

# Exercise 3: which promise did they break?

This is the exercise. A request can miss because it waited to start, or
because it generated slowly, and those are different bugs with different
fixes.

In [ ]:
# One 'p99 latency' number cannot tell you what to repair. Divide the misses
# by the promise that each one broke.
TTFT_SLA, TPOT_SLA = 1.0, 0.020
slow_start = 
slow_tokens = 
both = 
print(f'missed on TTFT only:  {100*slow_start.mean():5.1f}%  -> queueing or prefill')
print(f'missed on TPOT only:  {100*slow_tokens.mean():5.1f}%  -> the step is too slow')
print(f'missed on both:       {100*both.mean():5.1f}%')
# And in the TTFT misses, was the cause the queue or the prefill?

# Exercise 4: look at the shapes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
# Plot the three distributions, with the SLA lines on them.

plt.tight_layout()
plt.show()
print(f'TTFT    mean {ttft.mean():.3f}s  median {np.median(ttft):.3f}s  '
      f'p99 {np.percentile(ttft,99):.3f}s')
print(f'the mean is {ttft.mean()/np.median(ttft):.1f}x the median: a long right tail')

### Before you open the solution

1. Your p99 latency is a big number. Name the three different engineering
   projects it could be asking for, and say which of your other metrics
   distinguishes them.
2. Compare the mean TTFT with the median. Which would you put on a
   dashboard, and what would the other one hide?
3. Raw throughput did not move between the three SLA rows in Exercise 2,
   and goodput moved a lot. Which of the two is the server's capacity?